## 다중 선형 회귀 모델 - 스타트업 투자 문제
: 여러 요인 중 무엇이 결과를 움직이는지 밝혀내는 기본이 되는 AI모델<br>
: 여러 개의 독립변수가 하나의 종속변수에 얼마나 영향을 주는지 수식으로 표현된 모델<br>

**수식 구조**<br>
y = a₁x₁ + a₂x₂ + a₃x₃ + b<br>
y : 종속변수- 정답 - Label<br>
x₁, x₂, x₃ : 독립변수<br>
a₁, a₂, a₃ : 가중치 - 기울기<br>
b : 절편<br>

y → 예측하려는 값 (Profit)<br>
x₁, x₂, x₃ → 영향 요인 (R&D, Admin, Marketing, State)<br>
a₁, a₂, a₃ → 각 요인의 "영향력"<br>
b → 절편<br>

**데이터구성** <br>
연구개발비 - 숫자형
관리비용 - 숫자형
마케팅비용 - 숫자형
State - 범주형
이익 - 숫자형(종속변수-정답)

어떤 비용이 수익에 가장 영향을 많이 미치는가?<br>

**선택한 이유** <br>
다중 선형 회귀
원핫인코딩
특성 중요도 분석
과적합 분석

#### 실습 흐름
데이터 확인<br>
범주형 인코딩(state)<br>
X, y분리<br>
훈련데이터/테스트데이터 분리<br>
회귀 모델 학습<br>
평가<br>
Feature중요도 분석<br>

STEP#1 - 데이터 불러오기

In [161]:
import pandas as pd
df = pd.read_csv('../../data/50_Startups.csv')
df

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94
5,131876.90,99814.71,362861.36,New York,156991.12
6,134615.46,147198.87,127716.82,California,156122.51
7,130298.13,145530.06,323876.68,Florida,155752.60
8,120542.52,148718.95,311613.29,New York,152211.77
9,123334.88,108679.17,304981.62,California,149759.96


In [162]:
df.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [163]:
df.tail()

,R&D Spend,Administration,Marketing Spend,State,Profit
45,1000.23,124153.04,1903.93,New York,64926.08
46,1315.46,115816.21,297114.46,Florida,49490.75
47,0.00,135426.92,0.00,California,42559.73
48,542.05,51743.15,0.00,New York,35673.41
49,0.00,116983.80,45173.06,California,14681.40


In [164]:
df.describe()

,R&D Spend,Administration,Marketing Spend,Profit
count,50.000000,50.000000,50.000000,50.000000
mean,73721.615600,121344.639600,211025.097800,112012.639200
std,45902.256482,28017.802755,122290.310726,40306.180338
min,0.000000,51283.140000,0.000000,14681.400000
25%,39936.370000,103730.875000,129300.132500,90138.902500
50%,73051.080000,122699.795000,212716.240000,107978.190000
75%,101602.800000,144842.180000,299469.085000,139765.977500
max,165349.200000,182645.560000,471784.100000,192261.830000


In [165]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     str    
 4   Profit           50 non-null     float64
dtypes: float64(4), str(1)
memory usage: 2.1 KB


STEP#2 - X, y 분리

In [166]:
X = df.iloc[:, :-1]
X

,R&D Spend,Administration,Marketing Spend,State
0,165349.20,136897.80,471784.10,New York
1,162597.70,151377.59,443898.53,California
2,153441.51,101145.55,407934.54,Florida
3,144372.41,118671.85,383199.62,New York
4,142107.34,91391.77,366168.42,Florida
5,131876.90,99814.71,362861.36,New York
6,134615.46,147198.87,127716.82,California
7,130298.13,145530.06,323876.68,Florida
8,120542.52,148718.95,311613.29,New York
9,123334.88,108679.17,304981.62,California


In [167]:
y = df['Profit']
y

0     192261.83
1     191792.06
2     191050.39
3     182901.99
4     166187.94
5     156991.12
6     156122.51
7     155752.60
8     152211.77
9     149759.96
10    146121.95
11    144259.40
12    141585.52
13    134307.35
14    132602.65
15    129917.04
16    126992.93
17    125370.37
18    124266.90
19    122776.86
20    118474.03
21    111313.02
22    110352.25
23    108733.99
24    108552.04
25    107404.34
26    105733.54
27    105008.31
28    103282.38
29    101004.64
30     99937.59
31     97483.56
32     97427.84
33     96778.92
34     96712.80
35     96479.51
36     90708.19
37     89949.14
38     81229.06
39     81005.76
40     78239.91
41     77798.83
42     71498.49
43     69758.98
44     65200.33
45     64926.08
46     49490.75
47     42559.73
48     35673.41
49     14681.40
Name: Profit, dtype: float64

STEP#3 - 원핫인코딩(State컬럼)

In [ ]:
# 범주형 변수를 0/1 더미 변수로 바꿔줌 -> 원핫인코딩과 같은 개념임
# State열만 더미화하고 나머지 열은 그대로 유지함. 
# 첫번째 범주에 해당하는 더미 열을 제거함.
# drop_first=True: 빠진 한 범주를 기준 범주로 두고 모든 더미가 0일 때
# 그 기준 범주로 해석함
# (0,0)-> California, (1,0)-> Florida, (0,1)-> New York
# 더미를 k개가 아니라 k-1개만 만듦.
X = pd.get_dummies(X, columns=['State'], drop_first=True)
X

,R&D Spend,Administration,Marketing Spend,State_Florida,State_New York
0,165349.20,136897.80,471784.10,False,True
1,162597.70,151377.59,443898.53,False,False
2,153441.51,101145.55,407934.54,True,False
3,144372.41,118671.85,383199.62,False,True
4,142107.34,91391.77,366168.42,True,False
5,131876.90,99814.71,362861.36,False,True
6,134615.46,147198.87,127716.82,False,False
7,130298.13,145530.06,323876.68,True,False
8,120542.52,148718.95,311613.29,False,True
9,123334.88,108679.17,304981.62,False,False


STEP#4 - 훈련데이터/ 테스트 데이터 분리

In [169]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
# random_state = seed값

In [170]:
X_train

,R&D Spend,Administration,Marketing Spend,State_Florida,State_New York
33,55493.95,103057.49,214634.81,True,False
35,46014.02,85047.44,205517.64,False,True
26,75328.87,144135.98,134050.07,True,False
34,46426.07,157693.92,210797.67,False,False
18,91749.16,114175.79,294919.57,True,False
7,130298.13,145530.06,323876.68,True,False
14,119943.24,156547.42,256512.92,True,False
45,1000.23,124153.04,1903.93,False,True
48,542.05,51743.15,0.00,False,True
29,65605.48,153032.06,107138.38,False,True


In [171]:
X_test

,R&D Spend,Administration,Marketing Spend,State_Florida,State_New York
28,66051.52,182645.56,118148.20,True,False
11,100671.96,91790.61,249744.55,False,False
10,101913.08,110594.11,229160.95,True,False
41,27892.92,84710.77,164470.71,True,False
2,153441.51,101145.55,407934.54,True,False
27,72107.60,127864.55,353183.81,False,True
38,20229.59,65947.93,185265.10,False,True
31,61136.38,152701.92,88218.23,False,True
22,73994.56,122782.75,303319.26,True,False
4,142107.34,91391.77,366168.42,True,False


STEP#5 - 다중 선형 회귀 모델 생성 및 훈련

In [172]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


STEP#6 - 예측

In [173]:
y_pred = model.predict(X_test) # 예측한값 산출
y_pred

array([103015.20159796, 132582.27760816, 132447.73845174,  71976.09851258,
       178537.48221055, 116161.24230165,  67851.69209676,  98791.73374687,
       113969.43533012, 167921.0656955 ])

STEP#7 - 평가

In [174]:
# 회귀 모델 평가 지표( 결정계수, MSE )
# 결정계수: 실제 값의 분산 중 모델이 설명하는 비율. 0~1사이에 1에 가까울 수록 완벽 ( 1: 완벽. 과대적합 )
# MSE: ( 실제값 - 예측 )**2 -> 값이 작을 수록 예측이 정확 하다
from sklearn.metrics import r2_score, mean_squared_error
print('R2 Score: ', r2_score(y_test, y_pred) )
# 숫자가 큰 이유는 Profit자체가 수만 ~수십만 달러 단위라서 오차를 제곱하면 천만단위임
# 83502864를 루트를 씌우면 약 9100달러 정도 어긋나 있다라고 할 수 있음.
# Profit의 변동율이 약93%이므로 설명력과 적합도는 좋은 편임.
# MSE의 경우 절대 오차는 수천~수만 달러 일 수 있어 비즈니스 기준으로는 보통의 수준정도로 보여짐
print('MSE: ', mean_squared_error(y_test, y_pred))  # 데이터가 얼마나 분산되었는지 확인

R2 Score:  0.9347068473282423
MSE:  83502864.03257759


```txt
분산: 데이터가 평균으로부터 얼마나 퍼져 있는지를 수치로 나타낸 것
1반: 50, 50, 50, 50
2반: 30, 50, 70, 90

두 반 모두 평균운 50으로 동일하지만 2반이 훨씬 더 흩어져 있다
평균은 같으나 편차가 크다

분산이 크다: 데이터가 넓게 퍼져 있다. 변동성이 크다. 예측이 어렵다
분산이 작다: 평균 근처에 많이 몰려 있다. 안정적이다. 예측이 쉽다.
X분산이 크면 기울기가 안정적이다.
X의 분산이 거의 0이면 모델이 불안정하다
```

정리<br>
MSE는 얼마나 틀렸는가?에 해당되고 R2는 얼마나 설명했는가? 에 대한 수치이다<br>
만일 MSE는 낮은 데 R2도 낮은 경우<br>
R2는 높은데 MSE가 큰 경우<br>
변수를 20개 이상 넣었더니 R2가 0.99가 되었다면 -> 과적합 의심됨<br>

STEP#8 - 계수 해석
- 어떤 비용이 Profit에 가장 큰 영향을 주었나? -> ( Marketing Spend )

In [182]:
# 특성별 회귀계수를 DataFrame으로 정리 - 중요도 비교시 사용함
# X.column: 특성 이르미들임 - 여기서는 5개
# model.coef_ : 학습된 선형회귀의 계수
importance = pd.DataFrame({
  'Feature': X.columns,
  'Coefficient': model.coef_  
})
# Coefficient 내림차순 정렬 -> 영향이 큰 변수가 위로 옴.
print(importance.sort_values(by='Coefficient', ascending=False))

           Feature  Coefficient
4   State_New York   699.369053
0        R&D Spend     0.773467
2  Marketing Spend     0.036610
1   Administration     0.032885
3    State_Florida  -959.284160


```txt
           Feature  Coefficient
4   State_New York   699.369053 - 기준 범주에 비해 약 699 높음
0        R&D Spend     0.773467 - 연구 개발 비용 1단위 증가시 profit이 평균 0.77증가함
2  Marketing Spend     0.036610 - 마케팅 비용 1단위 증가시 profit이 평균 0.036증가함( 영향 적음 )
1   Administration     0.032885 - 관리비 1단위 증가시 profit이 평균 약 0.032 증가 ( 영향 적음 )
3    State_Florida  -959.284160
```